In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

####1. Rebuild employees table

In [0]:
%sql
DROP TABLE IF EXISTS employees;

CREATE OR REPLACE TABLE employees (
  employee_id  INT,
  name         STRING,
  department   STRING,
  salary       DOUBLE,
  status       STRING
)
USING DELTA
COMMENT 'Employee records';

INSERT INTO employees VALUES
  (1, 'Alice Nguyen',  'Engineering', 95000.00, 'active'),
  (2, 'Bob Patel',     'Engineering', 88000.00, 'active'),
  (3, 'Carol Santos',  'Engineering', 92000.00, 'active'),
  (4, 'David Kim',     'Engineering', 78000.00, 'active'),
  (5, 'Eva Müller',    'Marketing',   72000.00, 'active'),
  (6, 'Frank Osei',    'Marketing',   68000.00, 'active'),
  (7, 'Grace Lin',     'Marketing',   74000.00, 'active'),
  (8, 'Hiro Yamamoto', 'Marketing',   69000.00, 'active');

UPDATE employees
SET    salary = ROUND(salary * 1.10, 2)
WHERE  department = 'Engineering';

INSERT INTO employees VALUES
  (9, 'Ingrid Larsson', 'Engineering', 85000.00, 'active');

UPDATE employees
SET    status = 'terminated'
WHERE  employee_id = 6;

DESCRIBE HISTORY employees;

####2. INCIDENT 1: Bad UPDATE — missing WHERE clause

In [0]:
%sql
UPDATE employees
SET    salary = 999999.99;

SELECT employee_id, name, department, salary FROM employees ORDER BY employee_id;

####3. INCIDENT 2: Accidental DELETE — missing WHERE clause

In [0]:
%sql
DELETE FROM employees;

SELECT employee_id, name, department, salary FROM employees ORDER BY employee_id;

####4. Check history before restoring

In [0]:
%sql
DESCRIBE HISTORY employees;

####5. RESTORE to stable version 

In [0]:
%sql
--RESTORE TABLE employees TO TIMESTAMP AS OF '<V4-TIMESTAMP>';
RESTORE TABLE employees TO VERSION AS OF 6;

####6. Verify the restore

In [0]:
%sql
SELECT * FROM employees ORDER BY employee_id;

####7. View history showing RESTORE as a new commit

In [0]:
%sql
DESCRIBE HISTORY employees;

###Partial recovery: time travel + MERGE

####1. Corrupt Engineering salaries only

In [0]:
%sql
UPDATE employees
SET    salary = 1.00
WHERE  department = 'Engineering';

####2. Partial recovery with MERGE from historical version

In [0]:
%sql
MERGE INTO employees AS target
USING (
  SELECT employee_id, salary
  FROM   employees VERSION AS OF 9
  WHERE  department = 'Engineering'
) AS source
ON target.employee_id = source.employee_id
WHEN MATCHED THEN
  UPDATE SET target.salary = source.salary;

####3 Verify partial recovery

In [0]:
%sql
SELECT employee_id, name, department, salary, status
FROM   employees
ORDER  BY department, employee_id;